# Topic 1B: Pandas for Financial Data Wrangling
**Module 1 - Introduction to Machine Learning in Python**

This notebook demonstrates Pandas operations for financial data:
loading, cleaning, merging, time-series operations, and GroupBy aggregation.


## Setup
Download the LendingClub dataset from Kaggle:
https://www.kaggle.com/datasets/wordsforthewise/lending-club

Or use the sample data generated below.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

# Generate sample loan data (replace with real LendingClub data if available)
np.random.seed(42)
n = 10000
df = pd.DataFrame({
    'loan_id': range(n),
    'loan_amnt': np.random.lognormal(9.5, 0.5, n).round(0),
    'int_rate': np.random.uniform(5, 25, n).round(2),
    'annual_income': np.random.lognormal(11, 0.7, n).round(0),
    'dti': np.random.uniform(0, 40, n).round(2),
    'grade': np.random.choice(['A','B','C','D','E','F','G'], n, p=[0.15,0.25,0.25,0.15,0.10,0.06,0.04]),
    'home_ownership': np.random.choice(['RENT','MORTGAGE','OWN','OTHER'], n, p=[0.40,0.42,0.15,0.03]),
    'purpose': np.random.choice(['debt_consolidation','credit_card','home_improvement','major_purchase','other'], n, p=[0.40,0.25,0.15,0.10,0.10]),
    'emp_length': np.random.choice([np.nan,0,1,2,3,5,7,10], n, p=[0.08,0.05,0.10,0.12,0.15,0.15,0.15,0.20]),
    'issue_d': pd.date_range('2018-01-01', periods=n, freq='3H'),
})

# Create default column (8% default rate, higher for worse grades)
grade_default_prob = {'A':0.03, 'B':0.05, 'C':0.08, 'D':0.12, 'E':0.18, 'F':0.25, 'G':0.35}
df['default'] = df['grade'].map(grade_default_prob).apply(lambda p: np.random.binomial(1, p))

# Introduce some missing values
df.loc[np.random.choice(n, 500, replace=False), 'annual_income'] = np.nan
df.loc[np.random.choice(n, 300, replace=False), 'dti'] = np.nan

print(f'Shape: {df.shape}')
print(f'Default rate: {df["default"].mean():.4f}')
df.head()


## 1. Data Inspection


In [ ]:
# Basic inspection - always start with these 3 commands
print('=== INFO ===')
print(df.info())
print('\n=== DESCRIBE ===')
print(df.describe().round(2))
print('\n=== MISSING DATA ===')
missing = df.isnull().sum()
missing = missing[missing > 0]
print(missing)
print(f'\nTotal rows with any missing: {df.isnull().any(axis=1).sum()}')


## 2. Missing Data Report


In [ ]:
missing_report = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().mean() * 100).round(2),
    'dtype': df.dtypes
}).sort_values('missing_pct', ascending=False)

missing_report = missing_report[missing_report['missing_pct'] > 0]
print('Missing Data Report:')
print(missing_report)


## 3. Default Rate by Segment (GroupBy)


In [ ]:
# Default rate by grade
by_grade = df.groupby('grade').agg(
    count=('default', 'count'),
    default_rate=('default', 'mean'),
    avg_income=('annual_income', 'mean'),
    avg_int_rate=('int_rate', 'mean'),
    avg_loan=('loan_amnt', 'mean')
).round(4)
print('Default Rate by Grade:')
print(by_grade)

print('\nDefault Rate by Home Ownership:')
print(df.groupby('home_ownership')['default'].agg(['count', 'mean']).round(4))

print('\nDefault Rate by Purpose:')
print(df.groupby('purpose')['default'].agg(['count', 'mean']).sort_values('mean', ascending=False).round(4))


## 4. Time-Series Operations


In [ ]:
# Set date index for time-series operations
ts = df.set_index('issue_d')

# Monthly loan volume
monthly_volume = ts.resample('M')['loan_amnt'].sum()

# Monthly default rate
monthly_default = ts.resample('M')['default'].mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.plot(monthly_volume / 1e6, color='steelblue')
ax1.set_ylabel('Loan Volume ($M)')
ax1.set_title('Monthly Loan Origination Volume')

ax2.plot(monthly_default, color='indianred')
ax2.axhline(y=monthly_default.mean(), color='black', linestyle='--', label=f'Mean: {monthly_default.mean():.3f}')
ax2.set_ylabel('Default Rate')
ax2.set_title('Monthly Default Rate')
ax2.legend()

plt.tight_layout()
plt.show()


## 5. Merging Tables (Simulating Bureau Data)


In [ ]:
# Create synthetic bureau data
np.random.seed(42)
bureau = pd.DataFrame({
    'loan_id': np.repeat(df['loan_id'].values, np.random.randint(0, 8, n)),
})
bureau['bureau_credit_amount'] = np.random.lognormal(10, 1, len(bureau)).round(0)
bureau['bureau_overdue_days'] = np.random.exponential(10, len(bureau)).round(0)
bureau['bureau_credit_active'] = np.random.choice(['Active', 'Closed'], len(bureau), p=[0.4, 0.6])

print(f'Bureau records: {len(bureau)}')
print(f'Unique loan_ids in bureau: {bureau["loan_id"].nunique()}')

# IMPORTANT: Aggregate BEFORE merging (one-to-many join problem)
bureau_agg = bureau.groupby('loan_id').agg(
    num_bureau_records=('bureau_credit_amount', 'count'),
    avg_bureau_credit=('bureau_credit_amount', 'mean'),
    max_overdue_days=('bureau_overdue_days', 'max'),
    num_active_credits=('bureau_credit_active', lambda x: (x == 'Active').sum()),
).reset_index()

print(f'\nAggregated bureau shape: {bureau_agg.shape}')

# Left join: keep all loans, add bureau features where available
df_merged = df.merge(bureau_agg, on='loan_id', how='left')

# Fill NaN for loans with no bureau records
df_merged['num_bureau_records'] = df_merged['num_bureau_records'].fillna(0)

print(f'Merged shape: {df_merged.shape}')
print(f'Loans with no bureau data: {(df_merged["num_bureau_records"] == 0).sum()}')


## 6. Feature Correlation with Target


In [ ]:
numeric_cols = df_merged.select_dtypes(include=[np.number]).columns
correlations = df_merged[numeric_cols].corrwith(df_merged['default']).abs().sort_values(ascending=False)
print('Features most correlated with default:')
print(correlations.head(10).round(4))
